# CVE/CWE → Attack Family Mapping

Dataset: `CVE_CWE_2025.csv`

## 1. Import Libraries

Start by importing what we need: pandas for data handling, plus the CWE hierarchy file.

In [ ]:
import tensorflow as tf
#import transformers
#from transformers import TFBertModel

In [ ]:
import pandas as pd
import numpy as np

from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

In [ ]:
cwe_df = pd.read_csv("/kaggle/input/datasets/stanislavvinokur/cve-and-cwe-dataset-1999-2025/CVE_CWE_2025.csv")
cwe_df.head()

In [ ]:
cwe_df["CWE-ID"].value_counts().head(30)

## 2. Build CWE/CVE → Attack Family Mapping

Attack family maps individual CWE-IDs into 13 broader attack families (Injection, XSS, 
Memory Corruption, etc.) based on the MITRE CWE hierarchy and frequency 
analysis of the top CWEs in our dataset.

In [ ]:
attack_family = {
    # Injection-related
    "CWE-89": "Injection",        # SQL Injection
    "CWE-78": "Injection",        # OS Command Injection
    "CWE-77": "Injection",        # Command Injection (general)
    "CWE-94": "Injection",        # Code Injection
    "CWE-74": "Injection",        # Injection (general/base)
    "CWE-20": "Injection",        # Improper Input Validation (judgment call)
    "CWE-918": "Injection",       # SSRF
    "CWE-611": "Injection",       # XXE
    "CWE-427": "Injection",       # Uncontrolled Search Path Element
    "CWE-601": "Injection",      
    
    # Cross-Site Scripting
    "CWE-79": "XSS",

    # Memory Corruption
    "CWE-119": "Memory Corruption",  # Buffer overflow (general)
    "CWE-787": "Memory Corruption",  # Out-of-bounds Write
    "CWE-125": "Memory Corruption",  # Out-of-bounds Read
    "CWE-416": "Memory Corruption",  # Use After Free
    "CWE-476": "Memory Corruption",  # NULL Pointer Dereference
    "CWE-120": "Memory Corruption",  # Buffer Copy w/o Checking Size
    "CWE-190": "Memory Corruption",  # Integer Overflow
    "CWE-121": "Memory Corruption",  # Stack-based Buffer Overflow
    "CWE-122": "Memory Corruption",  # Heap-based Buffer Overflow
    "CWE-189": "Memory Corruption",  # Numeric Errors

    # Info Disclosure
    "CWE-200": "Info Disclosure",
    "CWE-532": "Info Disclosure",    # Insertion of Sensitive Info into Log Files

    # CSRF
    "CWE-352": "CSRF",

    # Path Traversal
    "CWE-22": "Path Traversal",
    "CWE-59": "Path Traversal",      # Improper Link Resolution (symlink attacks)

    # Authentication & Access Control
    "CWE-264": "Authentication & Access Control",  # Permissions/Privileges/Access Control
    "CWE-284": "Authentication & Access Control",  # Improper Access Control
    "CWE-862": "Authentication & Access Control",  # Missing Authorization
    "CWE-863": "Authentication & Access Control",  # Incorrect Authorization
    "CWE-269": "Authentication & Access Control",  # Improper Privilege Management
    "CWE-287": "Authentication & Access Control",  # Improper Authentication
    "CWE-306": "Authentication & Access Control",  # Missing Authentication for Critical Function
    "CWE-732": "Authentication & Access Control",  # Incorrect Permission Assignment
    "CWE-798": "Authentication & Access Control",  # Hard-coded Credentials
    "CWE-276": "Authentication & Access Control",  # Incorrect Default Permissions
    "CWE-522": "Authentication & Access Control",  # Insufficiently Protected Credentials
    "CWE-639": "Authentication & Access Control",  # Authorization Bypass via User-Controlled Key
    "CWE-255": "Authentication & Access Control",  # Credentials Management Errors

    # File Handling
    "CWE-434": "File Handling",  # Unrestricted Upload of File with Dangerous Type

    # Denial of Service
    "CWE-400": "Denial of Service",  # Uncontrolled Resource Consumption
    "CWE-399": "Denial of Service",  # Resource Management Errors
    "CWE-770": "Denial of Service",  # Allocation of Resources Without Limits
    "CWE-401": "Denial of Service",  # Missing Release of Memory (memory leak)

    # Cryptographic Issues
    "CWE-310": "Cryptographic Issues",
    "CWE-295": "Cryptographic Issues",  # Improper Certificate Validation

    # Deserialization
    "CWE-502": "Deserialization",

    # Race Conditions
    "CWE-362": "Race Condition",

    # NVD's catch-all bucket
    "NVD-CWE-Other": "Other",
}

## 3. Apply the Mapping

Apply the dictionary to create a new `attack_family` column, with any 
unmapped CWE falling back to "Other".

In [ ]:
cwe_df["attack_family"] = cwe_df["CWE-ID"].map(attack_family).fillna("Other")
cwe_df["attack_family"].value_counts()

## 4. Checking some of the Labels

I realized that sometimes CWE labels may reflect the software weakness rather than the description, which can lead to inconsistencies between the description and the ID. So I was hoping the NLP would catch that and correct it.

In [ ]:
cwe_df[["DESCRIPTION", "attack_family"]].sample(10)

## 5. Define Features (X) and Target (y)

- **X**: the CVE description text (model input)
- **y**: the attack_family label (what we're predicting)

In [ ]:
X = cwe_df["DESCRIPTION"]
y = cwe_df["attack_family"]

## 6. Train/Test Split

Split the data 80/20, stratified by `attack_family` to preserve class 
proportions across both sets since we have a class imbalance.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 11. Experimenting: Excluding "other" row

In [ ]:
mapped_only = cwe_df[cwe_df["attack_family"] != "Other"]
X_mapped = mapped_only["DESCRIPTION"]
y_mapped = mapped_only["attack_family"]

In [ ]:
print(f"Rows before: {len(cwe_df)}")
print(f"Rows after excluding Other: {len(mapped_only)}")

In [ ]:
X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_mapped, y_mapped, test_size=0.2, random_state=42, stratify=y_mapped
)

### Training BERT 

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import ModelCheckpoint

import transformers
from tqdm.notebook import tqdm
from tokenizers import BertWordPieceTokenizer
from transformers import BertTokenizer

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-large-uncased')
def bert_encode(data, maximum_length):
    # Converts data to a list of strings if it is a Pandas Series
    text_list = data.tolist() if hasattr(data, 'tolist') else list(data)
    
    encoded = tokenizer(
        text_list,
        add_special_tokens=True,
        max_length=maximum_length,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='np' # Directly outputs NumPy arrays
    )
    return encoded['input_ids'], encoded['attention_mask']

In [ ]:
train_input_ids,train_attention_masks = bert_encode(X,100)

In [ ]:
def create_model(bert_model,input_ids,attention_masks):
    output = bert_model([input_ids,attention_masks])
    output = output[1]
    output = tf.keras.layers.Dense(32,activation='relu')(output)
    output = tf.keras.layers.Dropout(0.2)(output)
    output = tf.keras.layers.Dense(1,activation='sigmoid')(output)

    model = tf.keras.models.Model(inputs = [input_ids,attention_masks],outputs = output)
    model.compile(Adam(lr=1e-5),loss='crossentropy',metrics=['accuracy'])
    return model

In [ ]:
bert_model = TFBertModel.from_pretrained('bert-base-uncased')

In [ ]:
model = create_model(bert_model,train_input_ids,train_attention_masks)
model.summary()

In [ ]:
history = model.fit(
    [train_input_ids,train_attention_masks],
    y,
    validation_split=0.2,
    epochs=3,
    batch_size=10
)

## Summary

I tested a few variations before settling on this model on step 9:
- Baseline (unigrams, no class weighting): 0.74 accuracy
- Balanced class weights: 0.68 accuracy (higher recall on rare classes, 
  but much lower precision)
- Balanced + bigrams: 0.71 accuracy
- Bigrams only (final, above): 0.77 accuracy
- 12-class (Other excluded, step 11): 0.89 accuracy, 0.84 macro F1